# Is the solver valid?

External review asked for this before more modelling layers go on: an interface
where problems with known answers can be run by hand, and where a known change of
coefficients gives a known change in result.

The question splits in two, and the split is most of the value
(Roache; AIAA G-077):

- **Verification** — *is the math solved correctly?* No aircraft data involved. A
  failure here is a defect in the core.
- **Validation** — *is it the right math?* Coefficients → modes → trajectories
  against published values.

`PROJECT.md` §4 mixes them, and was almost entirely the second kind.

### What each check depends on

| Tier | Depends on | Ageing risk |
|---|---|---|
| **0** | nothing — pure mathematics | **none** |
| **1** | an analytic relation, not data | **none** |
| **2** | a source's own arithmetic, closed loop | **none** |
| **3** | a claim about the physical world | high — **not claimed here** |

Review asked whether a 1972 document is a source of error. For tier 2 it is not:
if CR-2144's derivatives were 10% away from the real aeroplane, this model must
*still* reproduce CR-2144's own transfer-function factors from CR-2144's own
derivatives. **This notebook validates the solver, not fidelity to a real 747.**
The genuine risks — the data being the *flexible* airframe against a rigid-body
model, the scan, the small-perturbation range — are recorded in §5, and none of
them is the publication date.


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: this notebook is executed by the test suite

import matplotlib.pyplot as plt
import numpy as np

import flightsim  # enables float64 before any array is made
from flightsim import trim, validation, verification
from flightsim.aircraft import CRUISE, REGISTRY
from flightsim.units import FT2M

# Caughey Eq. (5.48) works M = 0.25 at sea level; CR-2144 Table IX-2's header
# says 165 KTAS. A 0.2% difference, recorded in the design spec.
CAUGHEY_V = 279.1 * FT2M
approach = REGISTRY["boeing747_approach"]
cruise = REGISTRY["boeing747"]
print(f"747 approach, {CAUGHEY_V:.2f} m/s at sea level")

## Tier 0 — does the arithmetic work?

Nothing below takes an aircraft's published data as a reference.

`PROJECT.md` §4 records angular-momentum drift of 5.7e-13 over 60,000 steps.
That says the integrator is *conservative*, not that it is *right*: a scheme can
conserve H beautifully and still be second-order when it claims to be fourth.
Nothing in the project asserted the order until now.


In [ ]:
dts = np.array([0.2, 0.1, 0.05, 0.025])
errors, slope = verification.oscillator_refinement(dts)
print(f"harmonic oscillator, exact solution known: observed order {slope:.5f}")

dts6 = np.array([1.0 / 4, 1.0 / 8, 1.0 / 16, 1.0 / 32])
V, H = CRUISE["boeing747"]["airspeed"], CRUISE["boeing747"]["altitude"]
errors6, slope6 = verification.fixed_control_refinement(
    cruise, V, H, dts6, dt_ref=1.0 / 1024.0
)
print(f"real 6-DOF, fine-step reference:          observed order {slope6:.5f}")

fig, ax = plt.subplots(figsize=(6, 4))
for d, e, lab in ((dts, errors, "harmonic oscillator"), (dts6, errors6, "6-DOF 747")):
    ax.loglog(d, e, "o-", label=f"{lab}")
    ax.loglog(d, e[0] * (d / d[0]) ** 4.0, "--", alpha=0.5, label="slope 4 reference")
ax.set_xlabel("dt (s)"); ax.set_ylabel("error"); ax.legend(fontsize=8)
ax.set_title("Observed order of accuracy")
fig.tight_layout()

In [ ]:
# The 6-DOF window stops at dt = 1/32 for a measured reason. The aircraft cruises
# at 40,000 ft, so pos_ned is about [944, 0, -12184] and float64 resolves it to
# 2.7e-12 m. The discretisation error reaches that floor near 7e-11 m, and past
# dt = 1/128 refining makes the answer WORSE -- pairwise order -0.685 at 1/256.
# A window run through the floor fits partly to round-off and reads 3.82.
print("smallest fitted 6-DOF error:", f"{errors6[-1]:.3e} m")
print("round-off floor (measured): ~7e-11 m")
print("headroom:", f"{errors6[-1] / 7e-11:.0f}x")

In [ ]:
history = verification.newton_residual_history(V, H, cruise, iterations=6)
print("Newton trim convergence -- the exponent should roughly double each step")
print(f"{'iteration':>10s} {'residual':>14s}")
for i, r in enumerate(history):
    print(f"{i:>10d} {r:14.3e}")

In [ ]:
# Torque-free rotation of an asymmetric body has a closed form in Jacobi elliptic
# functions, so "conserved" can be upgraded to "correct". No aircraft here.
I1, I2, I3 = 1420.0, 4070.0, 4780.0
omega0 = np.array([0.6, 0.0, 0.9])
t = np.linspace(0.0, 3.0, 601)
exact = verification.torque_free_omega(I1, I2, I3, omega0, t)

fig, ax = plt.subplots(figsize=(6, 3.5))
for i, lab in enumerate(("p", "q", "r")):
    ax.plot(t, exact[i], label=f"{lab} (closed form)")
ax.set_xlabel("t (s)"); ax.set_ylabel("rad/s"); ax.legend(fontsize=8)
ax.set_title("Torque-free rotation, Landau & Lifshitz §37")
fig.tight_layout()
print("The integrator matches this to 1e-8 over 1500 steps (test_verification.py).")

In [ ]:
# Wind may enter through vel_rel and NOWHERE else. PROJECT.md section 2 names two
# ways to get that wrong, and they need DIFFERENT instruments. A steady wind
# catches the Coriolis substitution -- a uniform steady wind is just a change of
# inertial frame, so attitude and rates must not move. Only a TIME-VARYING wind
# can catch a spurious -m dW/dt term, because a steady field's material
# derivative is identically zero and the bug is invisible in it.
#
# Note an invariance assertion is the WRONG instrument for the second one. The
# air-relative velocity obeys the still-air equation PLUS a -C^T Wdot term, so
# two runs offset by W(0) genuinely must diverge. What works is a closed form:
# zero every aerodynamic coefficient and the thrust, and the wind has no
# legitimate route into the equations at all, so free fall is the exact answer
# and any departure from it IS the spurious term.
ff = verification.free_fall_through_a_swinging_wind(
    verification.without_aerodynamics(cruise)
)
print(f"uniform wind swinging to {ff.peak_wind:.2f} m/s, peak |dW/dt| "
      f"{ff.peak_dwdt:.2f} m/s^2 = {ff.peak_dwdt / 9.80665:.2f} g")
print(f"departure from free fall over {ff.elapsed:.1f} s: "
      f"{ff.max_position_error:.2e} m")

# FALSIFIED, since a check that can only pass demonstrates nothing. With the
# spurious term injected into integrate.step -- differencing the cached previous
# wind against the current sample, the one line anyone would write -- the figure
# above becomes 13.33 m, ten orders of magnitude above the 1e-9 m the test
# asserts. The steady-wind test above passes with that bug still in, once the
# wind cache is seeded consistently, which is why this cell has to exist.


## Tier 1 — does a known coefficient change give the known result?

This is review's request read literally. Each sweep re-trims the aircraft, so the
mode and the trim point are not confounded.

**Every one of these relations turned out to be affine with a non-zero
intercept**, and the intercept is the term the textbook approximation drops. Three
of the four laws originally planned were the wrong functional form; the model was
right each time.


In [ ]:
def ph_zeta(a, alpha, de, thr):
    return validation.longitudinal_modes(a, alpha, de, thr, CAUGHEY_V, 0.0)[0][1]

base_CD0 = float(approach.CD0)
cd0s = base_CD0 * np.array([1.0, 1.5, 2.0, 3.0])
zetas = validation.sweep(approach, "CD0", cd0s, ph_zeta, CAUGHEY_V, 0.0)
slope, intercept, worst = validation.affine_fit(cd0s, zetas)
theory = 1.0 / (np.sqrt(2.0) * validation.REFERENCES["747pa_CL"].value)

print(f"zeta_phugoid vs CD0: slope {slope:.5f}, worst residual {worst * 100:.2f}%")
print(f"  textbook 1/(sqrt(2) CL) = {theory:.5f}  ->  model is {slope / theory:.2f}x")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(cd0s, zetas, "o", label="model")
ax.plot(cd0s, intercept + slope * cd0s, "-", alpha=0.6, label="linear fit")
ax.set_xlabel("CD0"); ax.set_ylabel(r"$\zeta_{phugoid}$"); ax.legend(fontsize=8)
ax.set_title("More drag damps the phugoid, linearly")
fig.tight_layout()

In [ ]:
def sp_wn(a, alpha, de, thr):
    return validation.longitudinal_modes(a, alpha, de, thr, CAUGHEY_V, 0.0)[-1][0]

cmas = np.array([-1.26, -0.9, -0.6, -0.3, -0.1])
wns = validation.sweep(approach, "Cma", cmas, sp_wn, CAUGHEY_V, 0.0)
slope_m, intercept_m, worst_m = validation.affine_fit(cmas, wns**2)
print(f"wn_sp^2 vs Cma: slope {slope_m:.5f}, intercept {intercept_m:.5f}, "
      f"worst residual {worst_m * 100:.2f}%")
print("The intercept is Z_alpha M_q / u0, which does NOT vanish at the neutral")
print("point -- so wn_sp does not go to zero there, and asserting it would have")
print("been asserting the approximation rather than the model.")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(cmas, wns**2, "o", label="model")
ax.plot(cmas, intercept_m + slope_m * cmas, "-", alpha=0.6, label="affine fit")
ax.axhline(intercept_m, ls=":", color="gray", label="intercept at Cma = 0")
ax.set_xlabel(r"$C_{m\alpha}$"); ax.set_ylabel(r"$\omega_{n,sp}^2$")
ax.legend(fontsize=8); ax.set_title("Short-period frequency vs pitch stiffness")
fig.tight_layout()

In [ ]:
# The neutral point itself: Cm_alpha = 0 is its definition, so the system must be
# neutrally stable there and divergent beyond. Nothing was tuned for this.
for cma in (-0.1, 0.0, 0.1):
    swept = approach._replace(Cma=__import__("jax").numpy.array(float(cma)))
    x, _ = trim.trim(
        __import__("jax").numpy.array(CAUGHEY_V),
        __import__("jax").numpy.array(0.0), swept)
    A = validation.longitudinal_matrix(
        swept, float(x[0]), float(x[1]), float(x[2]), CAUGHEY_V, 0.0)
    mx = max(lam.real for lam in np.linalg.eigvals(A))
    verdict = "DIVERGENT" if mx > 1e-6 else ("neutral" if abs(mx) <= 1e-6 else "stable")
    print(f"  Cma = {cma:+.2f}   max Re(lambda) = {mx:+.6f}   {verdict}")

## Tier 2 — against a published worked example

D. A. Caughey's Cornell MAE 5070 notes work CR-2144's 747 power-approach
condition and publish the whole chain: coefficients → dimensional derivatives →
plant matrix → characteristic polynomial → roots.

He is **not an independent data source** — his Eq. (5.48)–(5.50) cite Heffley &
Jewell, the same document this project transcribed. He *is* an independent
**implementation**: same inputs, different code, published intermediates. For
checking a solver that is the useful kind of independence.

The comparison needs an axis transform. Caughey states Θ₀ = 0, which is only true
in **stability axes**; the model linearises in **body axes**, where θ₀ = α₀ and
w₀ = V·sin α₀ ≠ 0. A rotation by α₀ is a similarity transform, so it moves every
element and leaves the eigenvalues alone.


In [ ]:
import jax.numpy as jnp

x, res = trim.trim(jnp.array(CAUGHEY_V), jnp.array(0.0), approach)
alpha, de, thr = float(x[0]), float(x[1]), float(x[2])
print(f"trim residual {float(jnp.linalg.norm(res)):.2e}, alpha {np.degrees(alpha):.3f} deg")

A_body = validation.longitudinal_matrix(approach, alpha, de, thr, CAUGHEY_V, 0.0)
A = validation.to_imperial_matrix(validation.to_stability_axes(A_body, alpha))
C = validation.CAUGHEY_A

names = [["Xu", "Xw", "Xq", "-g cos"], ["Zu", "Zw", "u0+Zq", "-g sin"],
         ["Mu'", "Mw'", "Mq'", "-"], ["-", "-", "1", "-"]]
print(f"\n{'element':16s} {'model':>12s} {'Caughey':>12s} {'rel':>9s}")
for i in range(4):
    for j in range(4):
        m, c = A[i, j], C[i, j]
        rel = f"{abs(m - c) / abs(c):9.4f}" if abs(c) > 1e-9 else f"{'--':>9s}"
        print(f"A[{i},{j}] {names[i][j]:<9s} {m:12.5f} {c:12.5f} {rel}")

In [ ]:
# The two elements that disagree by more than 3% carry the alpha-dot derivatives
# this model excludes by design. They are not merely attributed -- they are
# RECONSTRUCTED from Caughey's own tabulated CL_alphadot and Cm_alphadot.
Zwdot = validation.REFERENCES["747pa_Zwdot"].value
Mwdot = validation.REFERENCES["747pa_Mwdot"].value

print("Caughey's Z row is divided throughout by (1 - Zwdot):")
print(f"  A[1,1] {A[1, 1]:.5f} / {1 - Zwdot:.4f} = {A[1, 1] / (1 - Zwdot):.5f}"
      f"   vs published {C[1, 1]:.5f}")
print(f"  A[1,2] {A[1, 2]:.4f} / {1 - Zwdot:.4f} = {A[1, 2] / (1 - Zwdot):.4f}"
      f"   vs published {C[1, 2]:.4f}")
print("\nAnd this model's A[2,2] IS his raw Eq. (5.51) Mq:")
print(f"  model {A[2, 2]:.5f}  vs Eq. (5.51) Mq {validation.REFERENCES['747pa_Mq'].value:.5f}")
print(f"  + (u0+Zq)*Mwdot = {A[2, 2] + C[1, 2] * Mwdot:.5f}  vs published {C[2, 2]:.5f}")

In [ ]:
(ph_wn, ph_z), (sp_wn, sp_z) = validation.longitudinal_modes(
    approach, alpha, de, thr, CAUGHEY_V, 0.0)
R = validation.REFERENCES

rows = [("phugoid wn", ph_wn, "747pa_phugoid_wn"),
        ("phugoid zeta", ph_z, "747pa_phugoid_zeta"),
        ("short-period wn", sp_wn, "747pa_short_period_wn"),
        ("short-period zeta", sp_z, "747pa_short_period_zeta")]
print(f"{'mode':>20s} {'model':>10s} {'published':>11s} {'error':>8s}   source")
for label, model, key in rows:
    ref = R[key]
    err = abs(model - ref.value) / abs(ref.value)
    print(f"{label:>20s} {model:10.5f} {ref.value:11.5f} {err * 100:7.2f}%   {ref.source}")

In [ ]:
# The same code with the same omissions reads 17.8% at CRUISE. So the gap is not a
# fixed modelling deficit -- it is condition-dependent, and it bites at M 0.8 /
# 40,000 ft where compressibility drives the Mach content of Xu and Zu.
cruise_err = abs(R["747cruise_phugoid_wn_model"].value
                 - R["747cruise_phugoid_wn_ref"].value) / R["747cruise_phugoid_wn_ref"].value
approach_err = abs(ph_wn - R["747pa_phugoid_wn"].value) / R["747pa_phugoid_wn"].value
print(f"phugoid wn error at cruise   (M 0.80, 40,000 ft): {cruise_err * 100:5.2f}%")
print(f"phugoid wn error on approach (M 0.25, sea level): {approach_err * 100:5.2f}%")
print(f"ratio: {cruise_err / approach_err:.0f}x")

## What this establishes, and what it does not

**Established.** The integrator is fourth-order — asserted for the first time, on
a closed-form problem and through the real dynamics. A uniform wind only
translates the trajectory, which is the Coriolis-substitution error §2 warns
about and which still air structurally cannot see. A *time-varying* uniform wind
adds no body force either, against a closed-form free fall — the other error §2
names, and the seam a turbulence model will load. The trim Newton solve
converges quadratically. Torque-free rotation matches its analytic trajectory,
not merely its invariants. Every longitudinal plant-matrix element the model
contains matches an independent implementation of the same source data, and the
elements it omits are reconstructed from that source's own α̇ derivatives.

**Not established, and not claimed.** That this behaves like a real Boeing 747.
That needs flight-test data the project does not hold. Every tier-2 statement is
closed-loop against a document's own arithmetic, which is exactly why the
document's age does not threaten it.

**Known gaps, recorded in `PROJECT.md` §5 and §8.** CR-2144's 747 derivatives are
the *flexible* airframe and this is a rigid-body model. The 6-DOF error floor is
round-off at ~7e-11 m, below which refinement makes the answer worse. Gravity is
constant, which biases the phugoid by 0.38% at cruise altitude — measured, and
deliberately not corrected, because it consumes 7.6% of that mode's tolerance and
every result is quoted at one altitude per aircraft.
